# Synthetic Data Trust Lab: offline GPU run (TabDDPM, GReaT, Safe-Synthesizer)

Runs the GPU-only generators on Kaggle or Colab, evaluates them, and exports a results bundle (`.zip`)
that you import into your local store. Enable a GPU runtime (Kaggle: Settings > Accelerator; Colab:
Runtime > Change runtime type) and internet access, then run all cells top to bottom.

Steps: clone repo, install uv, build envs (CUDA torch for synthcity), `prepare`, `generate --allow-gpu-only`, `evaluate`, `export-results`.


In [ ]:
# Parameters. DATASET / max_rows must match what you registered locally (the import checks a dataset fingerprint).
REPO_URL = "https://github.com/rohitkumar0601/Synode.git"
DATASET = "adult"
SEEDS = "[0,1,2]"
GENERATORS = "[tabddpm,great,great_qwen,nss_smollm3,nss_smollm3_dp_eps8]"
EXTRA_OVERRIDES = []   # e.g. ["max_rows=20000", "budget=library"]; each becomes a Hydra `-o` override


In [ ]:
import os
import subprocess
import sys


def sh(cmd, **kw):
    """Run a shell command, stream its output, and raise on failure (unlike a bare `!cmd`)."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash", **kw)

if os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK = "/content"
else:
    raise RuntimeError("Neither /kaggle/working nor /content found: run this on Kaggle or Colab.")
print("working dir:", WORK)
sh("nvidia-smi")  # fails here if no GPU runtime is attached

## 1. Clone the repo

In [ ]:
REPO_DIR = f"{WORK}/Synode"
if not os.path.isdir(REPO_DIR):
    sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
sh("git log -1 --oneline")

## 2. Install uv and set storage locations

`STL_VENV_ROOT` is where `scripts/envrun.sh` puts the per-worker environments; `STL_STORAGE_ROOT` is the artifact store.
uv downloads a managed Python 3.11 for the synthcity env if the notebook's own Python is not 3.11.

In [ ]:
sh(f"{sys.executable} -m pip install -q uv")
os.environ["STL_VENV_ROOT"] = f"{WORK}/.stl"
os.environ["STL_STORAGE_ROOT"] = f"{WORK}/store"
OVERRIDES = " ".join(f"-o {o}" for o in [
    f"datasets={DATASET}", f"seeds={SEEDS}", f"generators_to_run={GENERATORS}", f"run_id={DATASET}-gpu",
    *EXTRA_OVERRIDES])
RUN = f"{DATASET}-gpu"

## 3. Build the environments, then swap synthcity's torch to CUDA 12.1

`workers/synthcity/pyproject.toml` pins `torch==2.2.2` to the CPU index (`pytorch-cpu`, an explicit named index).
uv does **not** let an environment variable override a named index URL (there is no `UV_INDEX_<NAME>_URL`;
only `UV_INDEX`, `UV_DEFAULT_INDEX` and `UV_INDEX_<NAME>_USERNAME/PASSWORD` exist). So instead we:

1. let `envrun.sh` create both envs (this installs `torch==2.2.2+cpu` into the synthcity env),
2. `uv pip install torch==2.2.2 --reinstall-package torch` from the CUDA 12.1 wheel index (`https://download.pytorch.org/whl/cu121`)
   into the synthcity env only (`--index-strategy unsafe-best-match` lets the `nvidia-*` dependencies come from PyPI),
3. export `UV_NO_SYNC=1` so later `uv run` calls (which `envrun.sh` uses) do not re-sync torch back to the CPU build.

The cell asserts that CUDA torch is active.

In [ ]:
sh("bash scripts/envrun.sh eval python -c 'import stl_eval; print(\"eval env ok\")'")
sh("bash scripts/envrun.sh synthcity python -c 'import torch; print(torch.__version__)'")   # creates env (cpu torch)
sh(f"uv pip install --python {WORK}/.stl/synthcity/bin/python 'torch==2.2.2' "
   "--index https://download.pytorch.org/whl/cu121 --index-strategy unsafe-best-match --reinstall-package torch")
os.environ["UV_NO_SYNC"] = "1"   # from here on `uv run` must not touch the (now CUDA) synthcity env
sh("bash scripts/envrun.sh synthcity python -c "
   "\"import torch; print(torch.__version__, torch.cuda.is_available()); "
   "assert torch.cuda.is_available() and 'cu121' in torch.__version__\"")

## 4. Prepare (register dataset, splits, plan jobs)

In [ ]:
sh(f"bash scripts/envrun.sh eval python experiments/run.py {OVERRIDES} prepare")

## 5. Generate (GPU-only generators, synthcity env)

In [ ]:
sh(f"bash scripts/envrun.sh synthcity python experiments/run.py {OVERRIDES} generate --allow-gpu-only")

## 6. Evaluate

In [ ]:
sh(f"bash scripts/envrun.sh eval python experiments/run.py {OVERRIDES} evaluate")

## 7. Export the results bundle

In [ ]:
ZIP = f"{WORK}/{RUN}.zip"
sh(f"bash scripts/envrun.sh eval python experiments/run.py {OVERRIDES} export-results --run {RUN} --out {ZIP}")
sh(f"unzip -l {ZIP}")

## 8. Import locally

Download the zip (Kaggle: Output tab or the file browser; Colab: Files panel, or run `from google.colab import files; files.download(ZIP)`).
Then, in your local repo:

```bash
bash scripts/envrun.sh eval python experiments/run.py -o datasets=<same dataset> import-results <path-to-zip>
```

The dataset must be registered locally with the **same `max_rows`** as on the notebook run, because the import checks the
dataset fingerprint (pass `--force` only if you accept a mismatch).